# nb_lib_retail

Shared PySpark functions for the silver, gold and data quality notebooks. They are loaded
with `%run nb_lib_retail`.

Every function here takes DataFrames and returns DataFrames. None of them reads or writes a
table, so the same code is unit tested locally with plain PySpark (see `tests/`).

In [ ]:
import datetime as dt
from functools import reduce

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

POS_COLUMNS = ["receipt_no", "line_no", "store_code", "sold_at", "sku", "quantity", "unit_price", "discount_pct", "currency"]

WEB_SCHEMA = T.StructType([
    T.StructField("order_id", T.StringType()),
    T.StructField("exported_at", T.StringType()),
    T.StructField("created_at", T.StringType()),
    T.StructField("market", T.StringType()),
    T.StructField("currency", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("lines", T.ArrayType(T.StructType([
        T.StructField("line_no", T.StringType()),
        T.StructField("sku", T.StringType()),
        T.StructField("qty", T.StringType()),
        T.StructField("unit_price", T.StringType()),
    ]))),
])

SILVER_SALES_COLUMNS = [
    "sales_line_id", "source", "order_id", "line_no", "store_code", "sku", "sold_at", "sale_date",
    "quantity", "unit_price_local", "discount_pct", "currency", "net_amount_local", "fx_rate_to_dkk",
    "fx_rate_source", "net_amount_dkk", "is_return", "order_status", "_version_ts", "_source_file",
    "_ingested_at", "_batch_id",
]

## Parsing

In [ ]:
def parse_number(c, dtype="decimal(18,2)"):
    """'1234,50', '1.234,50' and '1234.50' all become a decimal. Empty text becomes null."""
    s = F.trim(F.col(c) if isinstance(c, str) else c)
    s = F.when(s == "", None).otherwise(s)
    comma_decimal = s.rlike(r"^-?[0-9.]*,[0-9]+$")
    normalized = F.when(comma_decimal, F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", ".")).otherwise(s)
    return normalized.cast(dtype)


def _with_ids(df):
    return (df.withColumn("sale_date", F.to_date("sold_at"))
              .withColumn("sales_line_id", F.concat_ws(":", "source", "order_id", F.col("line_no").cast("string"))))


def clean_pos(df: DataFrame) -> DataFrame:
    """Bronze POS rows (all text) to typed sales lines. Store codes and SKUs are trimmed and upper-cased."""
    out = df.select(
        F.lit("pos").alias("source"),
        F.trim("receipt_no").alias("order_id"),
        F.col("line_no").cast("int").alias("line_no"),
        F.upper(F.trim("store_code")).alias("store_code"),
        F.to_timestamp(F.trim("sold_at"), "dd-MM-yyyy HH:mm").alias("sold_at"),
        F.upper(F.trim("sku")).alias("sku"),
        F.col("quantity").cast("int").alias("quantity"),
        parse_number("unit_price").alias("unit_price_local"),
        F.coalesce(parse_number("discount_pct"), F.lit(0).cast("decimal(18,2)")).alias("discount_pct"),
        F.upper(F.trim("currency")).alias("currency"),
        F.lit("completed").alias("order_status"),
        F.col("_ingested_at").alias("_version_ts"),
        "_source_file", "_ingested_at", "_batch_id",
        (F.col("store_code") != F.upper(F.trim("store_code"))).alias("_fixed_store_code"),
    )
    return _with_ids(out)


def flatten_web(df: DataFrame) -> DataFrame:
    """Bronze webshop orders (one row per export of an order) to one row per order line."""
    ts = "yyyy-MM-dd'T'HH:mm:ss'Z'"
    lines = df.select("*", F.explode_outer("lines").alias("l"))
    out = lines.select(
        F.lit("web").alias("source"),
        F.trim("order_id").alias("order_id"),
        F.col("l.line_no").cast("int").alias("line_no"),
        F.when(F.upper(F.trim("market")) == "DK", "WEB-DK").otherwise("WEB-EU").alias("store_code"),
        F.from_utc_timestamp(F.to_timestamp("created_at", ts), "Europe/Copenhagen").alias("sold_at"),
        F.upper(F.trim("l.sku")).alias("sku"),
        F.col("l.qty").cast("int").alias("quantity"),
        parse_number(F.col("l.unit_price")).alias("unit_price_local"),
        F.lit(0).cast("decimal(18,2)").alias("discount_pct"),
        F.upper(F.trim("currency")).alias("currency"),
        F.lower(F.trim("status")).alias("order_status"),
        F.to_timestamp("exported_at", ts).alias("_version_ts"),
        "_source_file", "_ingested_at", "_batch_id",
        F.lit(False).alias("_fixed_store_code"),
    )
    return _with_ids(out)


def latest_version(df: DataFrame, key="sales_line_id") -> DataFrame:
    """Keeps one row per key: the newest export wins. Identical re-sent rows collapse to one."""
    w = Window.partitionBy(key).orderBy(F.col("_version_ts").desc(), F.col("_ingested_at").desc(), F.col("_source_file").desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

## Master data

In [ ]:
def standardize_products(df: DataFrame) -> DataFrame:
    return df.select(
        F.upper(F.trim("sku")).alias("sku"),
        F.trim("product_name").alias("product_name"),
        F.initcap(F.trim("category")).alias("category"),
        F.trim("subcategory").alias("subcategory"),
        parse_number("list_price_dkk").alias("list_price_dkk"),
        parse_number("unit_cost_dkk").alias("unit_cost_dkk"),
        F.to_date(F.when(F.trim("launch_date") != "", F.trim("launch_date"))).alias("launch_date"),
        F.to_date(F.when(F.trim("discontinued_date") != "", F.trim("discontinued_date"))).alias("discontinued_date"),
    ).dropDuplicates(["sku"])


def standardize_stores(df: DataFrame) -> DataFrame:
    return df.select(
        F.upper(F.trim("store_code")).alias("store_code"),
        F.trim("store_name").alias("store_name"),
        F.trim("city").alias("city"),
        F.trim("country").alias("country"),
        F.trim("region").alias("region"),
        F.trim("channel").alias("channel"),
        F.upper(F.trim("currency")).alias("currency"),
        F.to_date(F.when(F.trim("opened_on") != "", F.trim("opened_on"))).alias("opened_on"),
        F.to_date(F.when(F.trim("closed_on") != "", F.trim("closed_on"))).alias("closed_on"),
    ).dropDuplicates(["store_code"])


def standardize_targets(df: DataFrame) -> DataFrame:
    return df.select(
        F.to_date("month").alias("month_start"),
        F.trim("city").alias("city"),
        parse_number("target_dkk").alias("target_dkk"),
    )


def fx_daily(fx: DataFrame, start, end) -> DataFrame:
    """One rate per currency per calendar day. Days without a published rate (weekends, holidays)
    use the last published rate before them. DKK is added with rate 1."""
    spark = fx.sparkSession
    rates = fx.select(
        F.to_date("rate_date").alias("rate_date"),
        F.upper(F.trim("currency")).alias("currency"),
        parse_number("dkk_per_unit", "decimal(18,6)").alias("published"),
    )
    cal = spark.createDataFrame([(start, end)], "s date, e date").select(F.explode(F.sequence("s", "e")).alias("rate_date"))
    grid = cal.crossJoin(rates.select("currency").distinct()).join(rates, ["rate_date", "currency"], "left")
    w = Window.partitionBy("currency").orderBy("rate_date").rowsBetween(Window.unboundedPreceding, 0)
    filled = grid.select(
        "rate_date", "currency",
        F.last("published", ignorenulls=True).over(w).alias("dkk_per_unit"),
        F.when(F.col("published").isNull(), "carried forward").otherwise("published").alias("rate_source"),
    )
    dkk = cal.select("rate_date", F.lit("DKK").alias("currency"), F.lit(1).cast("decimal(18,6)").alias("dkk_per_unit"),
                     F.lit("base currency").alias("rate_source"))
    return filled.unionByName(dkk)

## Validation and amounts

In [ ]:
QUARANTINE_REASONS = [
    "invalid_timestamp", "unknown_store", "before_store_opened", "after_store_closed", "unknown_sku",
    "invalid_quantity", "zero_quantity", "missing_price", "future_date", "no_fx_rate",
]


def validate_sales(df: DataFrame, stores: DataFrame, products: DataFrame, fx: DataFrame, run_date):
    """Splits sales lines into (valid, quarantine). Each quarantined row gets the first rule it breaks."""
    s = stores.select(F.col("store_code").alias("_s_code"), F.col("opened_on").alias("_opened"), F.col("closed_on").alias("_closed"))
    p = products.select(F.col("sku").alias("_p_sku"))
    x = fx.select(F.col("rate_date").alias("_fx_date"), F.col("currency").alias("_fx_cur"), "dkk_per_unit", "rate_source")
    j = (df.join(F.broadcast(s), df.store_code == s._s_code, "left")
           .join(F.broadcast(p), df.sku == p._p_sku, "left")
           .join(x, (df.sale_date == x._fx_date) & (df.currency == x._fx_cur), "left"))
    reason = (F.when(F.col("sold_at").isNull(), "invalid_timestamp")
               .when(F.col("_s_code").isNull(), "unknown_store")
               .when(F.col("sale_date") < F.col("_opened"), "before_store_opened")
               .when(F.col("_closed").isNotNull() & (F.col("sale_date") > F.col("_closed")), "after_store_closed")
               .when(F.col("_p_sku").isNull(), "unknown_sku")
               .when(F.col("quantity").isNull(), "invalid_quantity")
               .when(F.col("quantity") == 0, "zero_quantity")
               .when(F.col("unit_price_local").isNull(), "missing_price")
               .when(F.col("sale_date") > F.lit(run_date).cast("date"), "future_date")
               .when(F.col("dkk_per_unit").isNull(), "no_fx_rate"))
    j = j.withColumn("dq_reason", reason)
    cols = df.columns
    valid = j.where("dq_reason is null").select(*cols, "dkk_per_unit", "rate_source")
    quarantine = j.where("dq_reason is not null").select(*cols, "dq_reason")
    return valid, quarantine


def add_amounts(df: DataFrame) -> DataFrame:
    """Net amount in local currency after discount, converted to DKK with the day's rate (to the øre)."""
    gross = F.col("quantity") * F.col("unit_price_local")
    net_local = F.round(gross * (F.lit(1) - F.col("discount_pct") / 100), 2).cast("decimal(18,2)")
    return (df.withColumn("net_amount_local", net_local)
              .withColumn("fx_rate_to_dkk", F.col("dkk_per_unit"))
              .withColumn("fx_rate_source", F.col("rate_source"))
              .withColumn("net_amount_dkk", F.round(F.col("net_amount_local") * F.col("dkk_per_unit"), 2).cast("decimal(18,2)"))
              .withColumn("is_return", F.col("quantity") < 0)
              .select(*SILVER_SALES_COLUMNS))


def silver_sales_from_bronze(pos_raw, web_raw, stores, products, fx, run_date):
    """The whole bronze to silver step for one batch. Returns (valid, quarantine, stats)."""
    parts = []
    if pos_raw is not None:
        parts.append(clean_pos(pos_raw))
    if web_raw is not None:
        parts.append(flatten_web(web_raw))
    incoming = reduce(lambda a, b: a.unionByName(b), parts)
    deduped = latest_version(incoming)
    valid, quarantine = validate_sales(deduped, stores, products, fx, run_date)
    return add_amounts(valid), quarantine, incoming, deduped

## Gold star schema

In [ ]:
def surrogate_key(*cols):
    """Stable 64-bit key from business key columns, so keys do not change between runs."""
    return F.xxhash64(*[F.coalesce(F.col(c).cast("string"), F.lit("~")) for c in cols])


def date_key(c):
    return F.date_format(c, "yyyyMMdd").cast("int")


def build_dim_date(spark, start, end) -> DataFrame:
    df = spark.createDataFrame([(start, end)], "s date, e date").select(F.explode(F.sequence("s", "e")).alias("date"))
    fy = F.when(F.month("date") >= 6, F.year("date")).otherwise(F.year("date") - 1)
    return df.select(
        date_key("date").alias("date_key"),
        "date",
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month_no"),
        F.date_format("date", "MMM").alias("month_name"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.trunc("date", "month").alias("month_start"),
        F.concat(F.lit("FY"), F.substring(fy.cast("string"), 3, 2), F.lit("/"), F.substring((fy + 1).cast("string"), 3, 2)).alias("fiscal_year"),
        ((F.month("date") + 6) % 12 + 1).alias("fiscal_month_no"),
        F.date_format("date", "EEE").alias("weekday_name"),
        F.weekofyear("date").alias("iso_week"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
    )


def build_dim_city(stores: DataFrame) -> DataFrame:
    g = stores.groupBy("city").agg(
        F.countDistinct("country").alias("_n"), F.first("country").alias("_country"), F.first("region").alias("region"))
    return g.select(
        surrogate_key("city").alias("city_key"), "city",
        F.when(F.col("_n") > 1, "Multiple").otherwise(F.col("_country")).alias("country"), "region")


def build_dim_store(stores: DataFrame, as_of) -> DataFrame:
    closed = F.col("closed_on").isNotNull() & (F.col("closed_on") < F.lit(as_of).cast("date"))
    return stores.select(
        surrogate_key("store_code").alias("store_key"), "store_code", "store_name",
        surrogate_key("city").alias("city_key"), "city", "country", "region", "channel", "currency",
        "opened_on", "closed_on", F.when(closed, "Closed").otherwise("Open").alias("store_status"))


def build_dim_product(products: DataFrame, as_of) -> DataFrame:
    gone = F.col("discontinued_date").isNotNull() & (F.col("discontinued_date") < F.lit(as_of).cast("date"))
    return products.select(
        surrogate_key("sku").alias("product_key"), "sku", "product_name", "category", "subcategory",
        "list_price_dkk", "unit_cost_dkk", "launch_date", "discontinued_date",
        F.when(gone, "Discontinued").otherwise("Active").alias("product_status"))


def build_fact_sales(sales: DataFrame, products: DataFrame) -> DataFrame:
    """Completed sales only (cancelled and pending web orders stay in silver). Grain: one order line."""
    p = products.select("sku", "unit_cost_dkk")
    f = sales.where(F.col("order_status") == "completed").join(F.broadcast(p), "sku", "left")
    cost = F.round(F.col("quantity") * F.col("unit_cost_dkk"), 2).cast("decimal(18,2)")
    return f.select(
        "sales_line_id",
        date_key("sale_date").alias("date_key"),
        surrogate_key("store_code").alias("store_key"),
        surrogate_key("sku").alias("product_key"),
        "source", "order_id", "line_no", "quantity", "currency", "net_amount_local",
        F.col("net_amount_dkk").alias("net_revenue_dkk"),
        cost.alias("cost_dkk"),
        (F.col("net_amount_dkk") - cost).cast("decimal(18,2)").alias("gross_profit_dkk"),
        "is_return",
    )


def build_fact_target(targets: DataFrame) -> DataFrame:
    return targets.select(
        date_key("month_start").alias("date_key"),
        surrogate_key("city").alias("city_key"),
        F.col("target_dkk").cast("decimal(18,2)").alias("target_dkk"))

## Data quality rules

Each rule returns how many rows fail it. A **blocking** rule stops the pipeline, a
**warning** is recorded and shown in the report.

In [ ]:
def _anti(a, b, key):
    return a.join(b.select(key).distinct(), key, "left_anti").count()


def dq_rules(t: dict, run_date, max_quarantine_share=0.01, max_days_stale=2):
    """t holds the DataFrames: bronze_pos, bronze_web, silver_sales, quarantine, fact_sales,
    fact_target, dim_store, dim_product, dim_date, dim_city. Returns a list of result dicts."""
    res = []

    def add(rule_id, table, description, severity, failed, total, passed=None):
        ok = (failed == 0) if passed is None else passed
        res.append(dict(rule_id=rule_id, table_name=table, description=description, severity=severity,
                        failed_rows=int(failed), checked_rows=int(total), status="pass" if ok else "fail"))

    silver, q, fact = t["silver_sales"], t["quarantine"], t["fact_sales"]
    n_silver, n_q, n_fact = silver.count(), q.count(), fact.count()

    add("DQ01", "silver.sales_lines", "sales_line_id is unique", "blocking",
        n_silver - silver.select("sales_line_id").distinct().count(), n_silver)

    add("DQ02", "silver", "no line is both loaded and quarantined", "blocking",
        silver.select("sales_line_id").join(q.select("sales_line_id"), "sales_line_id").count(), n_silver)

    bronze_keys = (clean_pos(t["bronze_pos"]).select("sales_line_id")
                   .unionByName(flatten_web(t["bronze_web"]).select("sales_line_id")).distinct().count())
    add("DQ03", "bronze to silver", "every distinct source line is loaded or quarantined (row count reconciles)", "blocking",
        abs(bronze_keys - n_silver - n_q), bronze_keys)

    nulls = fact.where(F.col("date_key").isNull() | F.col("store_key").isNull() | F.col("product_key").isNull()
                       | F.col("net_revenue_dkk").isNull()).count()
    add("DQ04", "gold.fact_sales", "no null keys or amounts", "blocking", nulls, n_fact)
    add("DQ05", "gold.fact_sales", "every store_key exists in dim_store", "blocking", _anti(fact, t["dim_store"], "store_key"), n_fact)
    add("DQ06", "gold.fact_sales", "every product_key exists in dim_product", "blocking", _anti(fact, t["dim_product"], "product_key"), n_fact)
    add("DQ07", "gold.fact_sales", "every date_key exists in dim_date", "blocking", _anti(fact, t["dim_date"], "date_key"), n_fact)
    add("DQ08", "gold.fact_sales", "every sold product has a unit cost", "blocking", fact.where(F.col("cost_dkk").isNull()).count(), n_fact)

    s_sum = silver.where("order_status = 'completed'").agg(F.coalesce(F.sum("net_amount_dkk"), F.lit(0))).first()[0]
    g_sum = fact.agg(F.coalesce(F.sum("net_revenue_dkk"), F.lit(0))).first()[0]
    diff_ore = int(round(abs(float(s_sum) - float(g_sum)) * 100))
    add("DQ09", "silver to gold", "completed revenue in silver equals gold to the øre (failed_rows = difference in øre)",
        "blocking", diff_ore, n_fact)

    add("DQ10", "gold.fact_target", "every target city exists in dim_city", "blocking", _anti(t["fact_target"], t["dim_city"], "city_key"),
        t["fact_target"].count())

    share = n_q / max(n_silver + n_q, 1)
    add("DQ11", "silver.sales_quarantine", f"quarantined lines are at most {max_quarantine_share:.0%} of all lines", "warning",
        n_q, n_silver + n_q, passed=share <= max_quarantine_share)

    latest = silver.agg(F.max("sale_date")).first()[0]
    stale = (dt.date.fromisoformat(str(run_date)) - latest).days if latest else 999
    add("DQ12", "silver.sales_lines", f"newest sale is at most {max_days_stale} days old (failed_rows = days old)", "warning",
        max(stale, 0), n_silver, passed=stale <= max_days_stale)

    monthly = (fact.groupBy((F.col("date_key") / 100).cast("int").alias("ym"))
                   .agg((F.sum("gross_profit_dkk") / F.sum("net_revenue_dkk")).alias("gm")))
    add("DQ13", "gold.fact_sales", "monthly gross margin between 40% and 75%", "warning",
        monthly.where((F.col("gm") < 0.40) | (F.col("gm") > 0.75)).count(), monthly.count())
    return res